# HISTOPANTUM colorectal tumour classification

Binary tumour versus non-tumour classification using case-disjoint partitions. This notebook compares an ImageNet-pretrained ResNet50 with a frozen backbone against the same model with only its final `conv5` stage fine-tuned. It is a research and educational experiment, not a clinical system.

## Protocol

The TCGA case identifier is the independent grouping unit. All patches from one case stay in exactly one partition. The deterministic split optimizer targets 70/15/15 case counts while also balancing tumour and non-tumour patch counts. Training-only geometric and contrast augmentation is followed by ResNet50 Caffe preprocessing. Frozen and partially fine-tuned checkpoints are selected by validation loss, then each is evaluated exactly once on the untouched test set.

In [1]:
!unzip /content/histopantum.zip -d /content/histopantum

Streaming output truncated to the last 5000 lines.
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_15360.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_16384.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_27136.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16384_27648.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16896_13824.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_16896_28160.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_13312.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_14848.jpg  
  inflating: /content/histopantum/histopantum/colon/tumour/TCGA-NH-A6GC-01Z-00-DX1_17408_15872.jpg  
  inflating: /content/histopantum/histop

In [2]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from tensorflow import keras

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 15
HEAD_LEARNING_RATE = 1e-3
FINE_TUNE_LEARNING_RATE = 1e-5
SPLIT_SEARCH_ITERATIONS = 50_000
OUTPUT_DIR = Path('/content/exp6_outputs') if Path('/content').exists() else Path('outputs')
DATASET_ROOT = Path(os.environ.get('HISTOPANTUM_COLON_ROOT', '/content/histopantum/histopantum/colon'))

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print(f'Deterministic TensorFlow operations unavailable: {exc}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print({'python': sys.version, 'tensorflow': tf.__version__, 'sklearn': sklearn.__version__})


{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'tensorflow': '2.20.0', 'sklearn': '1.6.1'}


In [3]:
# In Colab, mount Drive before this cell or change HISTOPANTUM_COLON_ROOT above.
candidates = [
    DATASET_ROOT,
    Path('/content/histopantum/colon'),
    Path('/content/colon'),
    Path('data/histopantum/colon'),
]
DATASET_ROOT = next((path for path in candidates if path.is_dir()), DATASET_ROOT)
required = [DATASET_ROOT / 'non-tumour', DATASET_ROOT / 'tumour']
assert all(path.is_dir() for path in required), (
    f'Dataset not found at {DATASET_ROOT}. Expected non-tumour/ and tumour/.'
)
print('Dataset root:', DATASET_ROOT.resolve())


Dataset root: /content/histopantum/histopantum/colon


In [4]:
def parse_patch(path: Path, label: int) -> dict:
    """Parse a HISTOPANTUM filename into case, slide, and coordinates."""
    stem = path.stem
    parts = stem.rsplit('_', 2)
    if len(parts) != 3 or not parts[1].isdigit() or not parts[2].isdigit():
        raise ValueError(f'Unexpected patch filename: {path.name}')
    slide_id, x, y = parts
    case_parts = slide_id.split('-')
    if len(case_parts) < 3 or case_parts[0] != 'TCGA':
        raise ValueError(f'Unexpected TCGA slide identifier: {slide_id}')
    return {
        'path': str(path.resolve()), 'relative_path': path.relative_to(DATASET_ROOT).as_posix(),
        'label': label, 'class_name': 'tumour' if label else 'non-tumour',
        'case_id': '-'.join(case_parts[:3]), 'slide_id': slide_id,
        'x': int(x), 'y': int(y),
    }

records = []
for class_name, label in [('non-tumour', 0), ('tumour', 1)]:
    paths = sorted((DATASET_ROOT / class_name).glob('*.jpg'))
    assert paths, f'No JPEG files found for {class_name}'
    records.extend(parse_patch(path, label) for path in paths)
manifest = pd.DataFrame(records)
assert not manifest['relative_path'].duplicated().any()
assert manifest.groupby('case_id')['slide_id'].nunique().max() == 1, (
    'This notebook assumes one slide per case; revise grouping if the archive changes.'
)
assert set(manifest['label']) == {0, 1}
display(manifest.groupby('class_name').size().rename('patches').to_frame())
display(manifest.groupby('case_id').agg(patches=('label', 'size'), tumour=('label', 'sum')).describe())
print('Cases:', manifest['case_id'].nunique(), 'Slides:', manifest['slide_id'].nunique())


,patches
class_name,
non-tumour,10299
tumour,16949


,patches,tumour
count,40.000000,40.00000
mean,681.200000,423.72500
std,639.892747,432.90456
min,28.000000,0.00000
25%,173.000000,123.75000
50%,473.500000,287.50000
75%,918.000000,564.75000
max,2422.000000,1791.00000


Cases: 40 Slides: 40


In [5]:
def optimize_case_split(frame: pd.DataFrame, iterations: int, seed: int) -> dict[str, str]:
    """Find a deterministic case split close to target patch and class proportions."""
    case_table = frame.groupby('case_id')['label'].agg(patches='size', tumour='sum').reset_index()
    case_table['non_tumour'] = case_table['patches'] - case_table['tumour']
    cases = case_table['case_id'].to_numpy()
    values = case_table[['patches', 'non_tumour', 'tumour']].to_numpy(dtype=float)
    totals = values.sum(axis=0)
    targets = np.array([0.70, 0.15, 0.15])
    counts = [28, 6, 6]
    assert len(cases) == sum(counts), f'Expected 40 cases, found {len(cases)}'
    rng = np.random.default_rng(seed)
    best_score, best_order = np.inf, None
    for _ in range(iterations):
        order = rng.permutation(len(cases))
        groups = [order[:28], order[28:34], order[34:]]
        fractions = np.vstack([values[index].sum(axis=0) / totals for index in groups])
        if np.any(fractions[:, 1:] == 0):
            continue
        score = np.square((fractions - targets[:, None]) / targets[:, None]).mean()
        if score < best_score:
            best_score, best_order = score, order.copy()
    assert best_order is not None
    assignments = {}
    for split, indices in zip(('train', 'validation', 'test'), (best_order[:28], best_order[28:34], best_order[34:])):
        assignments.update({cases[index]: split for index in indices})
    print('Split optimization score:', best_score)
    return assignments

assignments = optimize_case_split(manifest, SPLIT_SEARCH_ITERATIONS, SEED)
manifest['split'] = manifest['case_id'].map(assignments)
assert not manifest['split'].isna().any()
case_sets = {name: set(part['case_id']) for name, part in manifest.groupby('split')}
assert case_sets['train'].isdisjoint(case_sets['validation'])
assert case_sets['train'].isdisjoint(case_sets['test'])
assert case_sets['validation'].isdisjoint(case_sets['test'])
summary = manifest.groupby('split').agg(
    patches=('label', 'size'), cases=('case_id', 'nunique'), slides=('slide_id', 'nunique'),
    tumour=('label', 'sum'),
)
summary['non_tumour'] = summary['patches'] - summary['tumour']
display(summary.loc[['train', 'validation', 'test']])
manifest.to_csv(OUTPUT_DIR / 'split_manifest.csv', index=False)


Split optimization score: 0.00040407383457137297


,patches,cases,slides,tumour,non_tumour
split,,,,,
train,19092,28,28,11868,7224
validation,3985,6,6,2450,1535
test,4171,6,6,2631,1540


In [6]:
AUTOTUNE = tf.data.AUTOTUNE

def decode_image(path: tf.Tensor, label: tf.Tensor):
    """Decode one JPEG as a fixed-shape float32 RGB tensor."""
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, tf.cast(label, tf.float32)

def make_dataset(part: pd.DataFrame, training: bool) -> tf.data.Dataset:
    """Create a finite deterministic dataset; augmentation lives in the model."""
    dataset = tf.data.Dataset.from_tensor_slices((part['path'].to_numpy(), part['label'].to_numpy()))
    if training:
        dataset = dataset.shuffle(len(part), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

parts = {name: manifest.loc[manifest['split'] == name].reset_index(drop=True) for name in ('train', 'validation', 'test')}
train_ds = make_dataset(parts['train'], training=True)
validation_ds = make_dataset(parts['validation'], training=False)
test_ds = make_dataset(parts['test'], training=False)
images, labels = next(iter(train_ds))
assert images.shape[1:] == (224, 224, 3) and labels.ndim == 1
assert bool(tf.reduce_all(tf.math.is_finite(images)))
print('Batch:', images.shape, labels.shape, 'range:', float(tf.reduce_min(images)), float(tf.reduce_max(images)))


Batch: (32, 224, 224, 3) (32,) range: 0.0 255.0


In [10]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
    keras.layers.RandomRotation(0.25, fill_mode='reflect', seed=SEED),
    keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED),
    keras.layers.RandomContrast(0.10, seed=SEED),
], name='training_augmentation')

backbone = keras.applications.ResNet50(
    include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3), pooling='avg'
)
backbone.trainable = False
inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
x = augmentation(inputs)
x = keras.layers.Lambda(keras.applications.resnet50.preprocess_input, name='caffe_preprocessing')(x)
x = backbone(x, training=False)
x = keras.layers.Dropout(0.30, seed=SEED)(x)
outputs = keras.layers.Dense(1, activation='sigmoid', name='tumour_probability')(x)
model = keras.Model(inputs, outputs, name='histopantum_crc_resnet50')

def compile_model(target: keras.Model, learning_rate: float) -> None:
    """Compile the binary classifier with reproducible metrics."""
    target.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'), keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall'), keras.metrics.AUC(name='roc_auc')],
    )

def load_checkpoint(path: Path) -> keras.Model:
    """Load checkpoints containing the named ResNet50 preprocessing function."""
    return keras.models.load_model(
        path,
        custom_objects={'preprocess_input': keras.applications.resnet50.preprocess_input},
    )

compile_model(model, HEAD_LEARNING_RATE)
probe = model(images[:2], training=False).numpy()
assert probe.shape == (2, 1) and np.isfinite(probe).all() and ((probe >= 0) & (probe <= 1)).all()
print('Frozen trainable parameters:', sum(np.prod(v.shape) for v in model.trainable_weights))


Frozen trainable parameters: 2049


In [8]:
FROZEN_CHECKPOINT = OUTPUT_DIR / 'resnet50_frozen_best.keras'
common_callbacks = lambda checkpoint: [
    keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_loss', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7, verbose=1),
]
frozen_history = model.fit(
    train_ds, validation_data=validation_ds, epochs=HEAD_EPOCHS,
    callbacks=common_callbacks(FROZEN_CHECKPOINT), verbose=1,
)
pd.DataFrame(frozen_history.history).to_csv(OUTPUT_DIR / 'frozen_history.csv', index_label='epoch')
assert FROZEN_CHECKPOINT.is_file()


Epoch 1/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.8661 - loss: 0.3071 - precision: 0.8797 - recall: 0.9092 - roc_auc: 0.9242
Epoch 1: val_loss improved from None to 0.61619, saving model to /content/exp6_outputs/resnet50_frozen_best.keras

Epoch 1: finished saving model to /content/exp6_outputs/resnet50_frozen_best.keras
597/597 ━━━━━━━━━━━━━━━━━━━━ 114s 171ms/step - accuracy: 0.9170 - loss: 0.2111 - precision: 0.9247 - recall: 0.9433 - roc_auc: 0.9710 - val_accuracy: 0.8168 - val_loss: 0.6162 - val_precision: 0.9103 - val_recall: 0.7788 - val_roc_auc: 0.8775 - learning_rate: 0.0010
Epoch 2/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step - accuracy: 0.9473 - loss: 0.1395 - precision: 0.9549 - recall: 0.9610 - roc_auc: 0.9870
Epoch 2: val_loss did not improve from 0.61619
597/597 ━━━━━━━━━━━━━━━━━━━━ 99s 166ms/step - accuracy: 0.9491 - loss: 0.1377 - precision: 0.9558 - recall: 0.9628 - roc_auc: 0.9871 - val_accuracy: 0.8156 - val_loss: 0.7277 - val_precision: 0.8964 - val

In [12]:
model = load_checkpoint(FROZEN_CHECKPOINT)
backbone = model.get_layer('resnet50')
backbone.trainable = True
for layer in backbone.layers:
    in_final_stage = layer.name.startswith('conv5_')
    layer.trainable = in_final_stage and not isinstance(layer, keras.layers.BatchNormalization)
trainable_backbone_layers = [layer.name for layer in backbone.layers if layer.trainable]
assert trainable_backbone_layers and all(name.startswith('conv5_') for name in trainable_backbone_layers)
assert not any(layer.trainable for layer in backbone.layers if isinstance(layer, keras.layers.BatchNormalization))
compile_model(model, FINE_TUNE_LEARNING_RATE)
print('Fine-tuned backbone layers:', len(trainable_backbone_layers))
print('Fine-tuning trainable parameters:', sum(np.prod(v.shape) for v in model.trainable_weights))

FINE_CHECKPOINT = OUTPUT_DIR / 'resnet50_conv5_best.keras'
fine_history = model.fit(
    train_ds, validation_data=validation_ds, epochs=FINE_TUNE_EPOCHS,
    callbacks=common_callbacks(FINE_CHECKPOINT), verbose=1,
)
pd.DataFrame(fine_history.history).to_csv(OUTPUT_DIR / 'fine_tune_history.csv', index_label='epoch')
assert FINE_CHECKPOINT.is_file()


Fine-tuned backbone layers: 22
Fine-tuning trainable parameters: 14955521
Epoch 1/15
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - accuracy: 0.9589 - loss: 0.1234 - precision: 0.9651 - recall: 0.9694 - roc_auc: 0.9884
Epoch 1: val_loss improved from None to 0.66996, saving model to /content/exp6_outputs/resnet50_conv5_best.keras

Epoch 1: finished saving model to /content/exp6_outputs/resnet50_conv5_best.keras
597/597 ━━━━━━━━━━━━━━━━━━━━ 147s 219ms/step - accuracy: 0.9634 - loss: 0.1051 - precision: 0.9692 - recall: 0.9719 - roc_auc: 0.9918 - val_accuracy: 0.8258 - val_loss: 0.6700 - val_precision: 0.9254 - val_recall: 0.7796 - val_roc_auc: 0.9094 - learning_rate: 1.0000e-05
Epoch 2/15
597/597 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - accuracy: 0.9732 - loss: 0.0712 - precision: 0.9794 - recall: 0.9776 - roc_auc: 0.9962
Epoch 2: val_loss improved from 0.66996 to 0.55854, saving model to /content/exp6_outputs/resnet50_conv5_best.keras

Epoch 2: finished saving model to /content/exp6_outputs/

In [13]:
checkpoints = {'frozen': FROZEN_CHECKPOINT, 'conv5_fine_tuned': FINE_CHECKPOINT}
validation_results = {}
for name, checkpoint in checkpoints.items():
    candidate = load_checkpoint(checkpoint)
    validation_results[name] = candidate.evaluate(validation_ds, return_dict=True, verbose=0)
display(pd.DataFrame(validation_results).T)
selected_name = min(validation_results, key=lambda name: validation_results[name]['loss'])
SELECTED_CHECKPOINT = OUTPUT_DIR / 'resnet50_selected_best.keras'
shutil.copy2(checkpoints[selected_name], SELECTED_CHECKPOINT)
print('Selected using validation loss only:', selected_name)
with (OUTPUT_DIR / 'validation_selection.json').open('w', encoding='utf-8') as handle:
    json.dump({'selected': selected_name, 'results': validation_results}, handle, indent=2)


,accuracy,loss,precision,recall,roc_auc
frozen,0.816813,0.616192,0.910305,0.778776,0.877518
conv5_fine_tuned,0.859975,0.485846,0.925360,0.840000,0.939343


Selected using validation loss only: conv5_fine_tuned


In [14]:
def evaluate_checkpoint(name: str, checkpoint: Path) -> dict:
    """Evaluate one prespecified checkpoint once and export patch/case evidence."""
    candidate = load_checkpoint(checkpoint)
    probabilities = candidate.predict(test_ds, verbose=1).reshape(-1)
    y_true = parts['test']['label'].to_numpy(dtype=int)
    y_pred = (probabilities >= 0.5).astype(int)
    assert len(probabilities) == len(y_true) and np.isfinite(probabilities).all()
    predictions = parts['test'][['relative_path', 'case_id', 'slide_id', 'label']].copy()
    predictions['probability'] = probabilities
    predictions['prediction'] = y_pred
    predictions.to_csv(OUTPUT_DIR / f'{name}_test_predictions.csv', index=False)
    pooled = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probabilities),
        'patches': len(y_true), 'cases': int(predictions['case_id'].nunique()),
    }
    case_rows = []
    for case_id, group in predictions.groupby('case_id'):
        true = group['label'].to_numpy(dtype=int)
        pred = group['prediction'].to_numpy(dtype=int)
        prob = group['probability'].to_numpy()
        case_rows.append({
            'case_id': case_id, 'patches': len(group), 'accuracy': accuracy_score(true, pred),
            'balanced_accuracy': balanced_accuracy_score(true, pred),
            'f1': f1_score(true, pred, zero_division=0),
            'roc_auc': roc_auc_score(true, prob) if len(np.unique(true)) == 2 else np.nan,
        })
    case_metrics = pd.DataFrame(case_rows)
    case_metrics.to_csv(OUTPUT_DIR / f'{name}_case_metrics.csv', index=False)
    aggregated = case_metrics[['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']].agg(['mean', 'std']).to_dict()
    case_macro = {metric: {stat: (None if pd.isna(value) else float(value)) for stat, value in stats.items()}
                  for metric, stats in aggregated.items()}
    result = {
        'checkpoint': checkpoint.name, 'threshold': 0.5, 'pooled_patch_metrics': pooled,
        'case_macro_metrics': case_macro, 'confusion_matrix': confusion_matrix(y_true, y_pred).tolist(),
        'classification_report': classification_report(
            y_true, y_pred, target_names=['non-tumour', 'tumour'], output_dict=True, zero_division=0
        ),
    }
    with (OUTPUT_DIR / f'{name}_test_metrics.json').open('w', encoding='utf-8') as handle:
        json.dump(result, handle, indent=2, allow_nan=False)
    return result

test_results = {name: evaluate_checkpoint(name, checkpoint) for name, checkpoint in checkpoints.items()}
comparison = pd.DataFrame({name: result['pooled_patch_metrics'] for name, result in test_results.items()}).T
comparison.to_csv(OUTPUT_DIR / 'test_model_comparison.csv', index_label='model')
display(comparison)
print('Deployment candidate selected before test evaluation:', selected_name)


131/131 ━━━━━━━━━━━━━━━━━━━━ 21s 145ms/step
131/131 ━━━━━━━━━━━━━━━━━━━━ 21s 143ms/step


,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,patches,cases
frozen,0.966675,0.966045,0.978495,0.968453,0.963636,0.973448,0.995331,4171.0,6.0
conv5_fine_tuned,0.967634,0.963708,0.970234,0.978715,0.948701,0.974456,0.996158,4171.0,6.0


Deployment candidate selected before test evaluation: conv5_fine_tuned


In [15]:
def sha256_file(path: Path) -> str:
    """Return the lowercase SHA-256 digest of a file."""
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

model_manifest = {
    'experiment_id': 'exp-6', 'dataset': 'HISTOPANTUM colorectal subset',
    'task': 'binary tumour versus non-tumour patch classification',
    'grouping_unit': 'TCGA case ID', 'seed': SEED, 'selected_phase': selected_name,
    'model_file': SELECTED_CHECKPOINT.name, 'model_size_bytes': SELECTED_CHECKPOINT.stat().st_size,
    'model_sha256': sha256_file(SELECTED_CHECKPOINT), 'input_shape': [224, 224, 3],
    'preprocessing': 'keras.applications.resnet50.preprocess_input (Caffe mode)',
    'decision_threshold': 0.5,
    'limitations': [
        'Only 40 TCGA cases are available.',
        'Patch-level observations within a case are correlated.',
        'This is internal colorectal evaluation, not domain or clinical validation.',
    ],
}
with (OUTPUT_DIR / 'model_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(model_manifest, handle, indent=2)
compact_dir = Path('/content/exp6_compact_files') if Path('/content').exists() else Path('exp6_compact_files')
if compact_dir.exists():
    shutil.rmtree(compact_dir)
compact_dir.mkdir(parents=True)
for artifact in OUTPUT_DIR.iterdir():
    if artifact.is_file() and artifact.suffix != '.keras':
        shutil.copy2(artifact, compact_dir / artifact.name)
archive_base = Path('/content/exp6_compact_evidence') if Path('/content').exists() else Path('exp6_compact_evidence')
archive = Path(shutil.make_archive(str(archive_base), 'zip', compact_dir, logger=None))
print('Selected model:', SELECTED_CHECKPOINT, model_manifest['model_sha256'])
print('Compact evidence:', archive, archive.stat().st_size, 'bytes')
print('Download the selected .keras checkpoint and compact evidence ZIP before ending Colab.')


Selected model: /content/exp6_outputs/resnet50_selected_best.keras 7fee95edb0df79a96ac4c0630d84ffa9e41c298292512dd850767087a8dbf00c
Compact evidence: /content/exp6_compact_evidence.zip 380557 bytes
Download the selected .keras checkpoint and compact evidence ZIP before ending Colab.


In [16]:
from google.colab import files

MODEL = "/content/exp6_outputs/resnet50_selected_best.keras"

import hashlib
from pathlib import Path

path = Path(MODEL)
print("Size:", path.stat().st_size)
print("SHA-256:", hashlib.sha256(path.read_bytes()).hexdigest())

files.download(MODEL)

Size: 214686627
SHA-256: 7fee95edb0df79a96ac4c0630d84ffa9e41c298292512dd850767087a8dbf00c


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!zip -r /content/exp6_outputs.zip /content/exp6_outputs


  adding: content/exp6_outputs/ (stored 0%)
  adding: content/exp6_outputs/conv5_fine_tuned_test_metrics.json (deflated 64%)
  adding: content/exp6_outputs/resnet50_conv5_best.keras (deflated 7%)
  adding: content/exp6_outputs/conv5_fine_tuned_case_metrics.csv (deflated 45%)
  adding: content/exp6_outputs/resnet50_selected_best.keras (deflated 7%)
  adding: content/exp6_outputs/conv5_fine_tuned_test_predictions.csv (deflated 91%)
  adding: content/exp6_outputs/frozen_history.csv (deflated 50%)
  adding: content/exp6_outputs/split_manifest.csv (deflated 95%)
  adding: content/exp6_outputs/resnet50_frozen_best.keras (deflated 8%)
  adding: content/exp6_outputs/validation_selection.json (deflated 51%)
  adding: content/exp6_outputs/test_model_comparison.csv (deflated 40%)
  adding: content/exp6_outputs/fine_tune_history.csv (deflated 52%)
  adding: content/exp6_outputs/model_manifest.json (deflated 40%)
  adding: content/exp6_outputs/frozen_test_predictions.csv (deflated 90%)
  adding: co